# Ⅰ第2回 部屋C 活性化関数の実験

**問い**：活性化関数を ReLU からシグモイドや tanh に替え，層を深くすると，勾配と正解率はどうなるか（第1回 GW1-4 の図）．

題材は第1回 GW3 試行①（FashionMNIST × MLP）．部屋Cのグループどうしで 6 つの条件を分担し，全員分で 1 枚の表を作る．
1 条件の実行時間の目安は 5 エポックで 1〜3 分．

## (0) グループと役割の設定

In [ ]:
# ===== (0) グループと役割の設定 =====
GROUP_ID = 1                  # ← 自分のグループ番号 (1〜27) に書き換える
MEMBER_ROLE = "implementer"   # implementer / verifier / recorder / presenter のいずれか（3 人・5 人グループの verifier は "verifier"）

# 授業用フォルダ（AI_TD）のルートを import パスに追加する（ノートブックをどこから開いても動く）
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".dlcourse_root").exists())
sys.path.insert(0, str(ROOT))
print("作業フォルダ:", ROOT)

## 予測（実行する前に書く）

このセルをダブルクリックして書き込む．

- 6 条件で正解率が一番高いのは：＿＿　一番低いのは：＿＿
- シグモイドで 6 層にしたとき，入力側の層1の勾配は出力側の何分の 1 になるか：＿＿

## (1) ライブラリのインポートと使用デバイス

In [ ]:
import math
import time
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from common.device import get_device, synchronize
from common.fashion import loaders            # GW3 試行①と同じデータ（訓練 48,000 枚・検証 12,000 枚）
from common.logger import ResultLogger
from common.todo import todo_check, quiz_check
plt.rcParams["font.family"] = ["Hiragino Sans", "sans-serif"]   # Mac の日本語フォント（文字化け防止）

device = get_device()
print("使用デバイス:", device)
from common.gradnorm import GradNormRecorder   # 部屋A で組み立てるものと同じ

## (確認) 実験の前に：用語の確認

授業ページの「勾配と学習率」を読んでから答える．**全問正解すると実験に進める**（ちがうとヒントが出る）．

**Q1**　勾配とは何か
- ア 損失の値そのもの
- イ 損失を小さくするには，重みをどちらへ，どれだけ動かせばよいかを表す数
- ウ 正解率がどれだけ上がったか

**Q2**　勾配が計算されるのは学習ループのどの行か
- ア outputs = model(images)
- イ loss.backward()
- ウ optimizer.step()

**Q3**　シグモイドの傾きは最大でも 0.25．層を重ねると，入力側の層の勾配はどうなりやすいか
- ア 大きくなる
- イ 小さくなる（消えかける）
- ウ 変わらない

In [ ]:
# ▼ 下の " " の中に，ア・イ・ウのどれかを書いて実行する
Q1 = ""
Q2 = ""
Q3 = ""
quiz_check({    # 2 つ目の値は答え合わせ用の暗号（答えは書いていない）
    "Q1": (Q1, "86c731cb", "勾配は，重みを直す向きと量を表す．損失そのものではない"),
    "Q2": (Q2, "af2c0979", "backward で勾配を計算し，step でその勾配を使って重みを直す"),
    "Q3": (Q3, "d0859473", "逆伝播では各層の傾きが掛け算されるので，0.25 以下を何回も掛けると小さくなる（勾配消失）"),
})

## (2) データ（GW3 試行①と同じ）

In [ ]:
torch.manual_seed(0)
train_loader, val_loader = loaders(batch_size=64)
images, labels = next(iter(train_loader))
print("ミニバッチの形:", tuple(images.shape), "| 1 エポックのステップ数:", len(train_loader))

## (3) 担当の条件

In [ ]:
ACT = "relu"        # ← "relu" / "sigmoid" / "tanh" のどれか
DEPTH = 2           # ← 隠れ層の数：2（浅い．GW3 試行①と同じ）か 6（深い）
if ACT not in ("relu", "sigmoid", "tanh") or DEPTH not in (2, 6):
    raise ValueError('ACT は "relu" / "sigmoid" / "tanh"，DEPTH は 2 か 6 にする')

## (4) モデル：活性化関数と深さを変えられる MLP

隠れ層はどれも幅 256．`DEPTH = 2` なら 全結合 → 活性化 → 全結合 → 活性化 → 出力 で，GW3 試行①とほぼ同じ形になる．

In [ ]:
ACTS = {"relu": nn.ReLU, "sigmoid": nn.Sigmoid, "tanh": nn.Tanh}

def make_mlp(act, depth, width=256):
    layers = [nn.Flatten()]
    in_features = 28 * 28
    for _ in range(depth):
        layers.append(nn.Linear(in_features, width))
        # TODO 1: 「...」を消して，活性化関数を 1 つ作って足す（ACTS["relu"]() は nn.ReLU() と同じ）
        layers.append(...)
        in_features = width
    layers.append(nn.Linear(in_features, 10))          # 出力層（活性化関数なし）
    return nn.Sequential(*layers)

todo_check("TODO 1")
torch.manual_seed(0)
model = make_mlp(ACT, DEPTH).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
model

## (5) 学習

In [ ]:
def validate(model):
    """検証データの損失と正解率（GW3 の⑪と同じ）"""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            total_loss += criterion(outputs, labels).item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total

EPOCHS = 5
recorder = GradNormRecorder(model)
train_losses, val_log = [], []

synchronize(device)
t0 = time.perf_counter()
for epoch in range(EPOCHS):
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        recorder.record()             # 勾配の大きさを記録する（backward の後，step の前）
        optimizer.step()
        train_losses.append(loss.item())
    val_loss, val_acc = validate(model)
    val_log.append((val_loss, val_acc))
    print(f"エポック {epoch + 1}/{EPOCHS}  訓練損失 {sum(train_losses[-len(train_loader):]) / len(train_loader):.4f}"
          f"  検証損失 {val_loss:.4f}  検証正解率 {val_acc * 100:.2f}%")
synchronize(device)
train_sec = time.perf_counter() - t0
grads = recorder.to_frame()
print(f"学習時間 {train_sec:.1f} 秒 | 記録したステップ数 {len(grads)}")

## (6) グラフ：層ごとの勾配と損失

表の「入力側の層1との比」が 1 より大きいほど，入力側の勾配が小さい（消えかけている）．

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
grads.rolling(50, min_periods=1).mean().plot(ax=axes[0], logy=True)
axes[0].set_title("層ごとの勾配の大きさ（50 ステップの移動平均）")
axes[0].set_xlabel("ステップ"); axes[0].set_ylabel("勾配の大きさ（対数目盛）")
pd.Series(train_losses).rolling(50, min_periods=1).mean().plot(ax=axes[1], color="black")
axes[1].set_title("訓練損失（50 ステップの移動平均）"); axes[1].set_xlabel("ステップ")
plt.tight_layout(); plt.show()

summary = pd.DataFrame({"最初の100ステップ": grads.head(100).mean(), "最後の100ステップ": grads.tail(100).mean()})
summary["入力側の層1との比"] = summary["最初の100ステップ"] / summary["最初の100ステップ"].iloc[0]
summary.round(4)

## (7) 記録

In [ ]:
metrics = {"val_loss": val_log[-1][0], "val_acc": val_log[-1][1], "train_sec": train_sec,
           "grad_start_first": grads.iloc[:100, 0].mean(), "grad_start_last": grads.iloc[:100, -1].mean(),
           "grad_end_first": grads.iloc[-100:, 0].mean(), "grad_end_last": grads.iloc[-100:, -1].mean()}
logger = ResultLogger(GROUP_ID, MEMBER_ROLE, course="c1", day="d2", exercise="room_c", device=device)
logger.log_many(metrics, condition=f"{ACT}_depth{DEPTH}", seed=0)
print("書き込み先:", logger.path)

## (共有) CSV を部屋の仲間と集める

1. 自分の `results/c1_d2_room_c_<グループ>.csv` を，同じ部屋の仲間全員に送る（Slack・AirDrop など）
2. 受け取った CSV は，すべて `results/room_c_shared/` フォルダに入れる（無ければ作る．同じ名前のファイルは末尾に `_2` などを付けて別名にする）
3. もう 1 条件担当するときは，担当のセルを書き換えて，そこから下をもう一度実行する（同じ CSV に行が増える）

## (8) 全員分の表（何回実行してもよい）

「出力側÷層1」は学習の最初の 100 ステップの勾配の比．大きいほど入力側の勾配が小さい．

In [ ]:
import glob
files = glob.glob(str(ROOT / "results" / "c1_d2_room_c_*.csv")) + glob.glob(str(ROOT / "results" / "room_c_shared" / "*.csv"))
df = pd.concat([pd.read_csv(f) for f in files]).drop_duplicates()
wide = df.pivot_table(index=["group_id", "condition"], columns="metric_name", values="metric_value").reset_index()
wide["出力側÷層1"] = wide["grad_start_last"] / wide["grad_start_first"]
table = wide.groupby("condition")[["val_acc", "val_loss", "出力側÷層1", "train_sec"]].mean()
print(f"集まった条件：{len(table)} / 6")
table.round(4)

## (9) 報告（提出）

このセルをダブルクリックして，＿＿ に書き込む（それぞれ 1〜2 文）．全体共有ではこの 3 つを 2 分で話す．

1. 深くしたとき，入力側の層1の勾配は出力側の何分の 1 になったか．活性化関数で違いはあったか：＿＿
2. 正解率が一番高い条件と低い条件．予測と比べてどうだったか：＿＿
3. 勾配が小さくなった条件でも，正解率は大きくは下がらなかったかもしれない．学習率を自動で調整する Adam を使っていることと関係があるか，考えを書く：＿＿